# XGBoost 随机种子搜索

`07_submission_xgboost_tuned_blend.csv` 是当前最好提交，核心方案是 `0.63 * Lasso + 0.37 * XGBoost(seed=42)`。

本 notebook 不改变数据处理和 XGBoost 参数，只搜索 XGBoost 的 `random_state`。由于 XGBoost 使用了行采样和列采样，不同随机种子会带来轻微不同的树结构，有时能改善泛化。

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LassoCV
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBRegressor

warnings.filterwarnings("ignore", category=UserWarning)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data"
SUBMISSION_DIR = ROOT / "submissions"

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

train.shape, test.shape

((1460, 81), (1459, 80))

## 1. 复用当前最佳数据处理

In [2]:
target = "SalePrice"
id_column = "Id"

outlier_mask = (train["GrLivArea"] > 4000) & (train["SalePrice"] < 300000)
train_model = train.loc[~outlier_mask].copy()

X = train_model.drop(columns=[target, id_column])
y = train_model[target]
X_test = test.drop(columns=[id_column]).copy()
test_ids = test[id_column]

print("移除异常点数量:", int(outlier_mask.sum()))
print("异常点 Id:", train.loc[outlier_mask, id_column].tolist())

移除异常点数量: 2
异常点 Id: [524, 1299]


In [3]:
def log_transform_skewed_features(X_train: pd.DataFrame, X_test: pd.DataFrame, threshold: float = 0.75):
    X_train = X_train.copy()
    X_test = X_test.copy()
    numeric_columns = X_train.select_dtypes(include="number").columns
    skewness = X_train[numeric_columns].skew().sort_values(ascending=False)
    skewed_columns = skewness[skewness > threshold].index.tolist()

    transformed_columns = []
    for column in skewed_columns:
        min_value = min(X_train[column].min(), X_test[column].min())
        if pd.notna(min_value) and min_value >= 0:
            X_train[column] = np.log1p(X_train[column])
            X_test[column] = np.log1p(X_test[column])
            transformed_columns.append(column)

    return X_train, X_test, transformed_columns


X_log, X_test_log, transformed_columns = log_transform_skewed_features(X, X_test)
print("log1p 处理的列数量:", len(transformed_columns))

log1p 处理的列数量: 20


## 2. 建模函数

In [4]:
numeric_features = X_log.select_dtypes(include="number").columns.tolist()
categorical_features = X_log.select_dtypes(exclude="number").columns.tolist()


def make_one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_model(base_model, scale_numeric: bool = False) -> TransformedTargetRegressor:
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline(numeric_steps), numeric_features),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", make_one_hot_encoder()),
                    ]
                ),
                categorical_features,
            ),
        ]
    )

    return TransformedTargetRegressor(
        regressor=Pipeline(steps=[("preprocessor", preprocessor), ("model", base_model)]),
        func=np.log1p,
        inverse_func=np.expm1,
    )


def rmsle(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_pred = np.maximum(y_pred, 0)
    return float(np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(y_pred))))

## 3. 生成 Lasso OOF

Lasso 保持不变，只搜索 XGBoost 的随机种子。

In [5]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

lasso_model = build_model(
    LassoCV(alphas=np.logspace(-4, 0, 40), cv=5, max_iter=30000, random_state=42),
    scale_numeric=True,
)

lasso_oof = np.zeros(len(X_log))
for train_idx, valid_idx in cv.split(X_log):
    fold_model = clone(lasso_model)
    fold_model.fit(X_log.iloc[train_idx], y.iloc[train_idx])
    lasso_oof[valid_idx] = fold_model.predict(X_log.iloc[valid_idx])

rmsle(y, lasso_oof)

0.11014232490544663

## 4. 搜索 XGBoost 随机种子

In [6]:
xgb_params = dict(
    n_estimators=1100,
    learning_rate=0.025,
    max_depth=3,
    subsample=0.80,
    colsample_bytree=0.70,
    reg_lambda=7,
    reg_alpha=0.01,
    objective="reg:squarederror",
    n_jobs=-1,
)

seeds = [0, 1, 2, 3, 5, 11, 17, 23, 31, 42, 73, 101, 123, 2024]
rows = []

for seed in seeds:
    xgb_model = build_model(XGBRegressor(random_state=seed, **xgb_params), scale_numeric=False)
    xgb_oof = np.zeros(len(X_log))

    for train_idx, valid_idx in cv.split(X_log):
        fold_model = clone(xgb_model)
        fold_model.fit(X_log.iloc[train_idx], y.iloc[train_idx])
        xgb_oof[valid_idx] = fold_model.predict(X_log.iloc[valid_idx])

    best_weight = None
    best_score = 999
    for w_lasso in np.arange(0.58, 0.701, 0.01):
        score = rmsle(y, w_lasso * lasso_oof + (1 - w_lasso) * xgb_oof)
        if score < best_score:
            best_weight = w_lasso
            best_score = score

    rows.append(
        {
            "seed": seed,
            "xgb_oof": rmsle(y, xgb_oof),
            "w_lasso": round(best_weight, 2),
            "w_xgboost": round(1 - best_weight, 2),
            "blend_oof": best_score,
        }
    )

seed_results = pd.DataFrame(rows).sort_values("blend_oof")
seed_results

,seed,xgb_oof,w_lasso,w_xgboost,blend_oof
7,23,0.114403,0.62,0.38,0.107478
10,73,0.114218,0.61,0.39,0.107487
12,123,0.114616,0.63,0.37,0.107584
9,42,0.114901,0.63,0.37,0.107594
13,2024,0.114703,0.63,0.37,0.107603
1,1,0.115025,0.63,0.37,0.107684
4,5,0.114763,0.63,0.37,0.107692
2,2,0.115038,0.64,0.36,0.107698
6,17,0.114956,0.63,0.37,0.107707
11,101,0.114912,0.63,0.37,0.107710


## 5. 生成提交文件

本地 OOF 最好的是 `seed=23`，融合权重为 `0.62 * Lasso + 0.38 * XGBoost`。

In [7]:
best = seed_results.iloc[0]
best_seed = int(best["seed"])
w_lasso = float(best["w_lasso"])
w_xgboost = float(best["w_xgboost"])

final_lasso = clone(lasso_model)
final_xgboost = build_model(
    XGBRegressor(random_state=best_seed, **xgb_params),
    scale_numeric=False,
)

final_lasso.fit(X_log, y)
final_xgboost.fit(X_log, y)

final_prediction = np.maximum(
    w_lasso * final_lasso.predict(X_test_log) + w_xgboost * final_xgboost.predict(X_test_log),
    0,
)

submission = pd.DataFrame({"Id": test_ids, "SalePrice": final_prediction})
SUBMISSION_DIR.mkdir(exist_ok=True)
submission_path = SUBMISSION_DIR / "09_submission_xgboost_seed_search.csv"
submission.to_csv(submission_path, index=False)

best_seed, w_lasso, w_xgboost, submission_path, submission.head(), submission["SalePrice"].describe()

(23,
 0.62,
 0.38,
 WindowsPath('C:/Users/84740/house-prices-advanced-regression/submissions/09_submission_xgboost_seed_search.csv'),
      Id      SalePrice
 0  1461  121333.518730
 1  1462  154815.632841
 2  1463  183995.234110
 3  1464  198115.583408
 4  1465  190956.801552,
 count      1459.000000
 mean     178679.172961
 std       78701.742824
 min       42110.415905
 25%      126555.591738
 50%      157518.076347
 75%      210251.513211
 max      860198.426333
 Name: SalePrice, dtype: float64)

## 6. 提交命令

```powershell
kaggle competitions submit -c house-prices-advanced-regression-techniques -f submissions/09_submission_xgboost_seed_search.csv -m "xgboost seed search lasso blend"
```